# Projeto 1 — Limpeza de Dados (ReclameAqui • Ibyte)

Este notebook executa o **tratamento inicial** do dataset bruto do ReclameAqui, com foco nos requisitos do Projeto 1 (Unidade I):

- Abertura do dataset no ambiente
- Tratamento inicial (nulos, padronização de textos, conversão de tipos)
- Criação de colunas auxiliares para análises e dashboard (tempo, localização, tamanho do texto)

**Entrada (raw):** `data/raw/RECLAMEAQUI_IBYTE.csv`  
**Saída (processado):** `data/prc/reclameaqui_ibyte_clean.csv`


## Checklist (Acompanhamento 1)

1. Definir objetivos de análise (no notebook de EDA)
2. Abrir o dataset no ambiente configurado
3. Demonstrar tratamento inicial:
   - valores ausentes (nulos)
   - padronização de textos
   - conversão de tipos


In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / "data").exists() and (parent / "docs").exists():
            return parent
    return start


In [2]:
PROJECT_ROOT = find_repo_root(Path.cwd())
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "RECLAMEAQUI_IBYTE.csv"
PRC_DIR = PROJECT_ROOT / "data" / "prc"
PRC_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH

WindowsPath('C:/Users/arthur.veras/pessoal/eda-projeto1/data/raw/RECLAMEAQUI_IBYTE.csv')

In [3]:
df_raw = pd.read_csv(
    RAW_PATH,
    encoding="utf-8",
    sep=",",
    skipinitialspace=True,
)

df_raw.shape

(1000, 16)

In [4]:
df_raw.head(3)

,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
0,21898349,Quebraram a tela do meu Notebook,Hidrolândia - CE,2016-02-11,Ibyte - Loja Física<->Informática,Resolvido,Em 05/10/2016 deixei meu Notebook para consert...,https://www.reclameaqui.com.br//ibyte-loja-fis...,2016,2,11,42,6,3,1,1
1,22581481,Solicitação de Cancelamento de Compra,Araxá - MG,2016-03-12,Não consigo fazer operação por telefone<->Cana...,Em réplica,"Prezados, efetuei uma compra no site www.ameri...",https://www.reclameaqui.com.br//ibyte-loja-fis...,2016,3,12,72,10,5,1,2
2,22579385,Inutilizarão a tela do meu notebook,Fortaleza - CE,2016-03-12,Problemas na Loja<->Ibyte - Loja Física<->Prod...,Respondida,Decidi conserta meu notebook que parou de inic...,https://www.reclameaqui.com.br//ibyte-loja-fis...,2016,3,12,72,10,5,1,2


In [5]:
# Diagnóstico inicial: tipos, nulos e duplicidades
display(df_raw.dtypes)

nulos = df_raw.isna().sum().sort_values(ascending=False)
display(nulos[nulos > 0])

print("Duplicados por ID:", df_raw.duplicated(subset=["ID"]).sum())

ID               int64
TEMA               str
LOCAL              str
TEMPO              str
CATEGORIA          str
STATUS             str
DESCRICAO          str
URL                str
ANO              int64
MES              int64
DIA              int64
DIA_DO_ANO       int64
SEMANA_DO_ANO    int64
DIA_DA_SEMANA    int64
TRIMETRES        int64
CASOS            int64
dtype: object

Series([], dtype: int64)

Duplicados por ID: 0


## Padronização e conversão de tipos

A seguir, vamos:

- padronizar nomes das colunas (snake_case)
- garantir tipos numéricos e temporais
- padronizar textos (trim, múltiplos espaços)
- extrair **cidade** e **UF** de `local`
- quebrar a hierarquia de `categoria` (separador `<->`)
- criar métricas de tamanho do texto (úteis para o dashboard)


In [6]:
RENAME = {
    "ID": "id",
    "TEMA": "tema",
    "LOCAL": "local",
    "TEMPO": "tempo",
    "CATEGORIA": "categoria",
    "STATUS": "status",
    "DESCRICAO": "descricao",
    "URL": "url",
    "ANO": "ano",
    "MES": "mes",
    "DIA": "dia",
    "DIA_DO_ANO": "dia_do_ano",
    "SEMANA_DO_ANO": "semana_do_ano",
    "DIA_DA_SEMANA": "dia_da_semana",
    "TRIMETRES": "trimestre",
    "CASOS": "casos",
}

df = df_raw.rename(columns=RENAME).copy()

# Tipos
text_cols = ["tema", "local", "tempo", "categoria", "status", "descricao", "url"]
num_cols = [
    "id",
    "ano",
    "mes",
    "dia",
    "dia_do_ano",
    "semana_do_ano",
    "dia_da_semana",
    "trimestre",
    "casos",
]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Data (TEMPO no dataset é uma data no formato YYYY-MM-DD)
df["data_reclamacao"] = pd.to_datetime(df["tempo"], errors="coerce")

df[["id", "data_reclamacao", "ano", "mes", "dia"]].head(3)

,id,data_reclamacao,ano,mes,dia
0,21898349,2016-02-11,2016,2,11
1,22581481,2016-03-12,2016,3,12
2,22579385,2016-03-12,2016,3,12


In [7]:
# Validação rápida: data_reclamacao bate com (ano, mes, dia)?
data_ymd = pd.to_datetime(
    df[["ano", "mes", "dia"]].rename(columns={"ano": "year", "mes": "month", "dia": "day"}),
    errors="coerce",
)

mismatch = (df["data_reclamacao"] != data_ymd).sum()
print("Datas inconsistentes:", int(mismatch))

Datas inconsistentes: 0


In [8]:
# Padronização do STATUS (garante consistência em dashboards)
status_map = {
    "resolvido": "Resolvido",
    "respondida": "Respondida",
    "em réplica": "Em réplica",
    "não resolvido": "Não resolvido",
    "não respondida": "Não respondida",
}

df["status"] = df["status"].str.lower().map(status_map).fillna(df["status"])
df["status"].value_counts()

status
Resolvido         430
Respondida        339
Em réplica        124
Não resolvido     103
Não respondida      4
Name: count, dtype: int64

In [9]:
# Extração de cidade e UF
df[["cidade", "uf"]] = df["local"].str.rsplit(" - ", n=1, expand=True)
df["cidade"] = df["cidade"].astype("string").str.strip()
df["uf"] = df["uf"].astype("string").str.strip().str.upper()

UF_TO_ESTADO = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins",
}

df["estado"] = df["uf"].map(UF_TO_ESTADO).astype("string")

df[["local", "cidade", "uf", "estado"]].head(3)

,local,cidade,uf,estado
0,Hidrolândia - CE,Hidrolândia,CE,Ceará
1,Araxá - MG,Araxá,MG,Minas Gerais
2,Fortaleza - CE,Fortaleza,CE,Ceará


In [10]:
# Quebra da hierarquia de categoria (separador <->)
cat_levels = df["categoria"].str.split("<->", expand=True)
cat_levels.columns = [f"categoria_n{i+1}" for i in range(cat_levels.shape[1])]
df = pd.concat([df, cat_levels], axis=1)

df["categoria_final"] = df["categoria"].str.split("<->").str[-1]
df["categoria_profundidade"] = df["categoria"].str.count("<->") + 1

df[["categoria", "categoria_n1", "categoria_n2", "categoria_n3", "categoria_n4", "categoria_n5", "categoria_final"]].head(3)

,categoria,categoria_n1,categoria_n2,categoria_n3,categoria_n4,categoria_n5,categoria_final
0,Ibyte - Loja Física<->Informática,Ibyte - Loja Física,Informática,<NA>,<NA>,<NA>,Informática
1,Não consigo fazer operação por telefone<->Cana...,Não consigo fazer operação por telefone,Canais de Atendimento,Ibyte - Loja Física,Problemas com o Atendimento,<NA>,Problemas com o Atendimento
2,Problemas na Loja<->Ibyte - Loja Física<->Prod...,Problemas na Loja,Ibyte - Loja Física,Produto danificado,Entrega,<NA>,Entrega


In [11]:
# Métricas do texto da reclamação (útil para filtros e análise)
df["descricao_n_caracteres"] = df["descricao"].str.len().astype("Int64")
df["descricao_n_palavras"] = df["descricao"].str.split().map(len).astype("Int64")

q33 = int(df["descricao_n_palavras"].quantile(1 / 3))
q66 = int(df["descricao_n_palavras"].quantile(2 / 3))

labels = [f"Curto (≤{q33})", f"Médio ({q33+1}–{q66})", f"Longo (>{q66})"]
df["faixa_tamanho_texto"] = pd.cut(
    df["descricao_n_palavras"],
    bins=[-1, q33, q66, float("inf")],
    labels=labels,
)

df[["descricao_n_palavras", "faixa_tamanho_texto"]].head(3)

,descricao_n_palavras,faixa_tamanho_texto
0,112,Médio (92–180)
1,122,Médio (92–180)
2,232,Longo (>180)


In [12]:
# Colunas temporais auxiliares
DIA_SEMANA_PT = {
    0: "Segunda",
    1: "Terça",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sábado",
    6: "Domingo",
}

df["dia_da_semana_nome"] = df["dia_da_semana"].map(DIA_SEMANA_PT).astype("string")
df["ano_mes"] = df["data_reclamacao"].dt.to_period("M").astype("string")

df[["data_reclamacao", "dia_da_semana", "dia_da_semana_nome", "ano_mes"]].head(3)

,data_reclamacao,dia_da_semana,dia_da_semana_nome,ano_mes
0,2016-02-11,3,Quinta,2016-02
1,2016-03-12,5,Sábado,2016-03
2,2016-03-12,5,Sábado,2016-03


In [13]:
# Qualidade pós-limpeza
display(df.isna().sum().sort_values(ascending=False).head(15))
print("IDs nulos:", int(df["id"].isna().sum()))
print("UFs inválidas:", int((~df["uf"].isin(UF_TO_ESTADO)).sum()))
print("Datas nulas:", int(df["data_reclamacao"].isna().sum()))
print("Registros:", len(df), "| Casos (soma):", int(df["casos"].sum()))

categoria_n5    983
categoria_n4    386
categoria_n3    298
categoria_n2    159
estado            8
id                0
local             0
tema              0
ano               0
mes               0
dia               0
tempo             0
categoria         0
status            0
descricao         0
dtype: int64

IDs nulos: 0
UFs inválidas: 8
Datas nulas: 0
Registros: 1000 | Casos (soma): 1600


In [14]:
OUT_PATH = PRC_DIR / "reclameaqui_ibyte_clean.csv"
df.to_csv(OUT_PATH, index=False, encoding="utf-8")

OUT_PATH

WindowsPath('C:/Users/arthur.veras/pessoal/eda-projeto1/data/prc/reclameaqui_ibyte_clean.csv')

## Próximos passos

- Rodar o notebook de EDA: `02_analise_exploratoria_reclameaqui_ibyte.ipynb`
- A partir do dataset limpo, desenhar as **métricas** e os **gráficos** que irão compor o dashboard
